In [31]:
# Install necessary libraries silently
!pip install datasets langdetect vaderSentiment nltk pyarrow --quiet

# Import core data manipulation libraries
import pandas as pd
import numpy as np
import re
import os

# Import libraries for dataset loading, language detection, and sentiment analysis
from datasets import load_dataset
from langdetect import detect
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Import NLTK for text processing and download required resources
import nltk
nltk.download("punkt")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

## STAGE 1 — **EXTRACTION**

In [32]:
# =====================================
# Load 50,000 conversations from WildChat-1M dataset
# =====================================

# Load the dataset in streaming mode for efficiency
ds = load_dataset(
    "allenai/WildChat-1M",
    split="train",
    streaming=True
)

records=[]

# Iterate through the dataset and collect the first 50,000 records
for i,row in enumerate(ds):

    records.append(row)

    if i >= 49999:
        break

# Convert the collected records into a pandas DataFrame
df_raw = pd.DataFrame(records)

# Print the shape of the raw DataFrame for verification
print(
    "Raw shape:",
    df_raw.shape
)

Raw shape: (50000, 14)


## STAGE 2 — **CLEANING**

In [33]:
# ================================
# Perform initial cleaning steps on the raw DataFrame
# ================================

# Drop duplicate conversations based on 'conversation_hash'
df_raw = df_raw.drop_duplicates(
   subset=["conversation_hash"]
)

# Fill missing 'country' values with 'UNKNOWN'
df_raw["country"] = (
   df_raw["country"]
   .fillna("UNKNOWN")
)

# Fill missing 'language' values with 'und' (undetermined)
df_raw["language"] = (
   df_raw["language"]
   .fillna("und")
)

# Convert 'toxic' column to integer type
df_raw["toxic"] = (
   df_raw["toxic"]
   .astype(int)
)

# Convert 'timestamp' column to datetime objects, coercing errors to NaT
df_raw["timestamp"] = pd.to_datetime(
   df_raw["timestamp"],
   errors="coerce",
   utc=True
)

# Print the shape of the DataFrame after cleaning
print("Shape after initial cleaning:", df_raw.shape)
print("\nDataFrame Info after cleaning:")
df_raw.info()

Shape after initial cleaning: (49550, 14)

DataFrame Info after cleaning:
<class 'pandas.core.frame.DataFrame'>
Index: 49550 entries, 0 to 49999
Data columns (total 14 columns):
 #   Column               Non-Null Count  Dtype              
---  ------               --------------  -----              
 0   conversation_hash    49550 non-null  object             
 1   model                49550 non-null  object             
 2   timestamp            49550 non-null  datetime64[ns, UTC]
 3   conversation         49550 non-null  object             
 4   turn                 49550 non-null  int64              
 5   language             49550 non-null  object             
 6   openai_moderation    49550 non-null  object             
 7   detoxify_moderation  49550 non-null  object             
 8   toxic                49550 non-null  int64              
 9   redacted             49550 non-null  bool               
 10  state                40882 non-null  object             
 11  country    

# Flatten **conversations**

In [34]:
message_rows=[]

# Iterate through each conversation in df_raw to flatten messages
for _,row in df_raw.iterrows():

    conv_id = row["conversation_hash"]

    # Iterate through individual messages within each conversation
    for idx,msg in enumerate(
        row["conversation"]
    ):

        # Append message details to message_rows list
        message_rows.append({

          "conversation_id":conv_id,

          "message_index":idx+1,

          "role":msg.get("role"),

          "content":msg.get("content"),

          # Use message timestamp if available, else use conversation timestamp
          "timestamp":(
             msg.get("timestamp")
             if msg.get("timestamp")
             else row["timestamp"]
          ),

          # Use message country if available, else use conversation country
          "country":(
             msg.get("country")
             if msg.get("country")
             else row["country"]
          ),

          # Use message language if available, else use conversation language
          "language":(
             msg.get("language")
             if msg.get("language")
             else row["language"]
          ),

          # Use message toxic flag if available, else use conversation toxic flag
          "toxic":(
             msg.get("toxic")
             if msg.get("toxic")
             else row["toxic"]
          ),

          "hashed_ip":row["hashed_ip"],

          "model":row["model"]

        })


# Create a new DataFrame from the flattened message rows
df_messages = pd.DataFrame(
   message_rows
)

# Print the shape of the messages DataFrame
print("Shape of flattened messages DataFrame:", df_messages.shape)
print("\nDataFrame Info for df_messages:")
df_messages.info()

Shape of flattened messages DataFrame: (290714, 10)

DataFrame Info for df_messages:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 290714 entries, 0 to 290713
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype              
---  ------           --------------   -----              
 0   conversation_id  290714 non-null  object             
 1   message_index    290714 non-null  int64              
 2   role             290714 non-null  object             
 3   content          290714 non-null  object             
 4   timestamp        290714 non-null  datetime64[ns, UTC]
 5   country          290714 non-null  object             
 6   language         290714 non-null  object             
 7   toxic            290714 non-null  int64              
 8   hashed_ip        290714 non-null  object             
 9   model            290714 non-null  object             
dtypes: datetime64[ns, UTC](1), int64(2), object(7)
memory usage: 22.2+ MB


# **STAGE 3 — TEXT PREPROCESSING**

In [35]:
# Initialize VADER Sentiment Intensity Analyzer
analyzer = SentimentIntensityAnalyzer()

# Define a function to clean text data
def clean_text(x):

    # Convert to string and lowercase
    x=str(x).lower()

    # Remove placeholders like [name] or [email]
    x = re.sub(
       r"\[name\]|\[email\]",
       "",
       x
    )

    # Replace multiple spaces with a single space and strip leading/trailing whitespace
    x = re.sub(
       r"\s+",
       " ",
       x
    ).strip()

    return x


# Apply the cleaning function to the 'content' column to create 'clean_text'
df_messages["clean_text"] = (
   df_messages["content"]
   .apply(clean_text)
)

# Calculate token count (number of words) for clean text
df_messages["token_count"] = (
   df_messages["clean_text"]
   .str.split()
   .apply(len)
)

# Calculate sentiment score using VADER for clean text
df_messages["sentiment_score"] = (
   df_messages["clean_text"]
   .apply(
      lambda x:
      analyzer.polarity_scores(x)["compound"]
   )
)

print("First 5 rows of cleaned text, token count, and sentiment scores:")
display(df_messages[['content', 'clean_text', 'token_count', 'sentiment_score']].head())

First 5 rows of cleaned text, token count, and sentiment scores:


,content,clean_text,token_count,sentiment_score
0,Hey there! Are you familiar with reality shift...,hey there! are you familiar with reality shift...,232,0.9399
1,Hey there! I'm more than happy to help you pla...,hey there! i'm more than happy to help you pla...,498,0.9982
2,Crea una imagen de una mujer corriente por la ...,crea una imagen de una mujer corriente por la ...,12,0.0000
3,"Como inteligencia artificial basada en texto, ...","como inteligencia artificial basada en texto, ...",133,-0.7003
4,Aceede a Amazon y dame una lista de lo TV más ...,aceede a amazon y dame una lista de lo tv más ...,18,0.1779


## Toxicity from moderation

In [36]:
# Function to calculate a single toxicity score from a list of moderation results
def moderation_toxic_score(mod_list):

    try:

      scores=[]

      # Iterate through each moderation item in the list
      for item in mod_list:

        # Get category scores, default to empty dict if not present
        cs=item.get(
           "category_scores",
           {}
        )

        # Extract scores for specific toxicity categories
        vals=[
          cs.get("hate",0),
          cs.get("harassment",0),
          cs.get("violence",0),
          cs.get("sexual",0)
        ]

        # Append the maximum score among the categories for this item
        scores.append(
           max(vals)
        )

      # Return the maximum score found across all moderation items
      return max(scores)

    except:
      # Return 0 if any error occurs during processing
      return 0


# Apply the moderation_toxic_score function to the 'openai_moderation' column
df_raw["toxicity_score"] = (
   df_raw["openai_moderation"]
   .apply(
      moderation_toxic_score
   )
)

# Create a binary flag 'toxic_flag' if toxicity_score exceeds a threshold (0.30)
df_raw["toxic_flag"] = (
   df_raw["toxicity_score"] > .30
).astype(int)

print("First 5 rows with toxicity_score and toxic_flag:")
display(df_raw[['openai_moderation', 'toxicity_score', 'toxic_flag']].head())

First 5 rows with toxicity_score and toxic_flag:


,openai_moderation,toxicity_score,toxic_flag
0,"[{'categories': {'harassment': False, 'harassm...",0.083071,0
1,"[{'categories': {'harassment': False, 'harassm...",0.070757,0
2,"[{'categories': {'harassment': False, 'harassm...",0.000202,0
3,"[{'categories': {'harassment': False, 'harassm...",0.000250,0
4,"[{'categories': {'harassment': False, 'harassm...",0.177658,0


## STAGE 4 — FEATURE **ENGINEERING**

In [37]:
# Aggregate message-level data to conversation-level features
conv = (
df_messages
.groupby("conversation_id")
.agg(

# Calculate total number of turns in a conversation
conv_turn_count=(
 "message_index",
 "max"
),

# Calculate average prompt length (token count) per conversation
avg_prompt_len=(
 "token_count",
 "mean"
),

# Calculate average sentiment score per conversation
avg_sentiment=(
 "sentiment_score",
 "mean"
),

# Get the first country, language, model, and hashed_ip for the conversation
country=("country","first"),

language=("language","first"),

model=("model","first"),

hashed_ip=("hashed_ip","first")

)
.reset_index()
)

# Merge 'conv' DataFrame with 'df_raw' to bring in toxicity scores
conv = conv.merge(
   df_raw[
    [
     "conversation_hash",
     "toxicity_score",
     "toxic_flag"
    ]
   ],
   left_on="conversation_id",
   right_on="conversation_hash"
)

# Drop the redundant 'conversation_hash' column
conv.drop(
 columns=["conversation_hash"],
 inplace=True
)

# Create a 'drop_off_flag' for conversations with 2 or fewer turns
conv["drop_off_flag"]=(
conv["conv_turn_count"]<=2
).astype(int)


# Calculate a 'response_quality_score' based on prompt length and toxicity
conv["response_quality_score"]=(
np.log1p(
conv["avg_prompt_len"]
)*2
-
conv["toxicity_score"]*3
)

print("Shape of conversation-level DataFrame:", conv.shape)
print("\nDataFrame Info for conv:")
conv.info()
print("\nFirst 5 rows of conv DataFrame:")
display(conv.head())

Shape of conversation-level DataFrame: (49550, 12)

DataFrame Info for conv:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 49550 entries, 0 to 49549
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   conversation_id         49550 non-null  object 
 1   conv_turn_count         49550 non-null  int64  
 2   avg_prompt_len          49550 non-null  float64
 3   avg_sentiment           49550 non-null  float64
 4   country                 49550 non-null  object 
 5   language                49550 non-null  object 
 6   model                   49550 non-null  object 
 7   hashed_ip               49550 non-null  object 
 8   toxicity_score          49550 non-null  float64
 9   toxic_flag              49550 non-null  int64  
 10  drop_off_flag           49550 non-null  int64  
 11  response_quality_score  49550 non-null  float64
dtypes: float64(4), int64(3), object(5)
memory usage: 4.5+ MB

First 5 r

,conversation_id,conv_turn_count,avg_prompt_len,avg_sentiment,country,language,model,hashed_ip,toxicity_score,toxic_flag,drop_off_flag,response_quality_score
0,000328f971437915462253ef7efa3d99,2,75.500,-0.038600,Russia,English,gpt-3.5-turbo-0301,03d3a8bcdcabd4bb19718415be4434e9260b6f5eafdfcf...,0.003256,0,1,8.664813
1,000389c3eb8f1a4d2c4aa7867d647393,2,114.000,0.449650,Lithuania,English,gpt-4-0314,22a4f96891b72dd26f13cfc8da44b5e72d23b4aba516ce...,0.000277,0,1,9.489035
2,0005a00b764ee8bec8e568719c979fe7,20,112.600,-0.040315,Italy,Italian,gpt-3.5-turbo-0301,332cd1b47fb8d0913529ab820a5bc616690cc1c68275e7...,0.004007,0,0,9.453345
3,00070b104d8fdfdb538683c30de86654,4,405.000,-0.499825,Germany,German,gpt-3.5-turbo-0301,ec0e475c818e34ecd3a61e69737f0e85df44b59ec7df53...,0.015944,0,0,11.964874
4,000712d3175ab7f12b9eec45da59b6b2,8,142.625,0.130975,Russia,Russian,gpt-4-0314,674476fe84e87fa33f374d2db98f909fd4322fb77919ff...,0.001905,0,0,9.928696


# Prompt **categories**

In [48]:
# Extract the first user prompt for each conversation
user_prompts=(
df_messages[
 df_messages["role"]=="user"
]
.sort_values(
["conversation_id","message_index"]
)
.groupby(
"conversation_id"
)
.first()
.reset_index()
)


# Define a function to classify user prompts into categories
def classify_prompt(text):

 t=str(text).lower()

 # Check for coding-related keywords
 if any(x in t for x in
 ["python","sql","code","bug"]):
   return "Coding"

 # Check for factual/informational keywords
 elif any(x in t for x in
 ["what","why","how","explain"]):
   return "Factual"

 # Check for creative writing keywords
 elif any(x in t for x in
 ["story","poem","write"]):
   return "Creative"

 # Classify short prompts as casual
 elif len(t.split()) < 8:
   return "Casual"

 # Classify prompts with a question mark as factual
 elif "?" in t:
   return "Factual"

 # Default category for other prompts
 else:
   return "Other"


# Apply the classification function to user prompts
user_prompts["prompt_category"]=(
user_prompts["content"]
.apply(classify_prompt)
)

# Before merging, remove any existing 'prompt_category' columns to avoid conflicts.
# This handles cases where the cell is re-run and the column might already exist.
for col in ['prompt_category', 'prompt_category_x', 'prompt_category_y']:
    if col in conv.columns:
        conv = conv.drop(columns=[col])

# Merge prompt categories back into the 'conv' DataFrame
conv = conv.merge(
user_prompts[
["conversation_id",
"prompt_category"]
],
on="conversation_id",
how="left" # Use left merge to keep all conversations in 'conv'
)

print("First 5 rows of conv DataFrame with prompt categories:")
display(conv[['conversation_id', 'prompt_category']].head())

First 5 rows of conv DataFrame with prompt categories:


,conversation_id,prompt_category
0,000328f971437915462253ef7efa3d99,Coding
1,000389c3eb8f1a4d2c4aa7867d647393,Other
2,0005a00b764ee8bec8e568719c979fe7,Other
3,00070b104d8fdfdb538683c30de86654,Other
4,000712d3175ab7f12b9eec45da59b6b2,Casual


# Daily **KPIs**

In [43]:
# Calculate daily KPIs for conversation count and average tokens
daily_kpis = (
df_messages
.groupby(
df_messages["timestamp"].dt.date # Group by date part of timestamp
)
.agg(

# Count unique conversations per day
conversation_count=(
"conversation_id",
"nunique"
),

# Calculate average token count per day
avg_tokens=(
"token_count",
"mean"
)

)
.reset_index()
)

# Calculate daily toxicity rate from the raw DataFrame
tox_daily=(
df_raw
.groupby(
df_raw["timestamp"].dt.date # Group by date part of timestamp
)
.agg(
# Calculate mean of toxic_flag to get toxicity rate
toxicity_rate=(
"toxic_flag",
"mean"
)
)
.reset_index()
)

# Merge daily_kpis with daily toxicity rates on timestamp
daily_kpis=daily_kpis.merge(
tox_daily,
on="timestamp"
)

print("Shape of daily KPIs DataFrame:", daily_kpis.shape)
print("\nDataFrame Info for daily_kpis:")
daily_kpis.info()
print("\nFirst 5 rows of daily_kpis DataFrame:")
display(daily_kpis.head())

Shape of daily KPIs DataFrame: (22, 4)

DataFrame Info for daily_kpis:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 22 entries, 0 to 21
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   timestamp           22 non-null     object 
 1   conversation_count  22 non-null     int64  
 2   avg_tokens          22 non-null     float64
 3   toxicity_rate       22 non-null     float64
dtypes: float64(2), int64(1), object(1)
memory usage: 836.0+ bytes

First 5 rows of daily_kpis DataFrame:


,timestamp,conversation_count,avg_tokens,toxicity_rate
0,2023-04-09,1796,128.043743,0.034110
1,2023-04-10,1909,93.482861,0.036997
2,2023-04-11,2103,90.150438,0.016561
3,2023-04-12,1903,91.416659,0.023874
4,2023-04-13,1979,88.840003,0.019291


## **STAGE 5 — OUTPUT CSVs**

In [44]:
# Save the cleaned conversations DataFrame to a CSV file
conv.to_csv(
"conversations_clean.csv",
index=False
)
print("Saved 'conversations_clean.csv'")

# Save the daily KPIs DataFrame to a CSV file
daily_kpis.to_csv(
"daily_kpis.csv",
index=False
)
print("Saved 'daily_kpis.csv'")

# Aggregate conversation data by country to create a geographical summary
geo_summary=(
conv.groupby("country")
.agg(
# Count total conversations per country
conversation_count=(
"conversation_id",
"count"
),
# Calculate toxicity rate per country
toxicity_rate=(
"toxic_flag",
"mean"
)
)
.reset_index()
)

# Save the geographical summary DataFrame to a CSV file
geo_summary.to_csv(
"geo_summary.csv",
index=False
)
print("Saved 'geo_summary.csv'")

print("\nFirst 5 rows of geo_summary DataFrame:")
display(geo_summary.head())

Saved 'conversations_clean.csv'
Saved 'daily_kpis.csv'
Saved 'geo_summary.csv'

First 5 rows of geo_summary DataFrame:


,country,conversation_count,toxicity_rate
0,Albania,2,0.000000
1,Algeria,230,0.000000
2,Argentina,226,0.000000
3,Armenia,11,0.000000
4,Australia,246,0.056911


# Combined Master **CSV**

In [45]:
# Create a combined DataFrame by copying 'conv'
combined=conv.copy()

# Add a 'date' column with the current UTC date
combined["date"]=pd.Timestamp.utcnow().date()

# Merge the combined DataFrame with daily_kpis based on the 'date' column
# Rename 'timestamp' in daily_kpis to 'date' for merging
combined=combined.merge(
daily_kpis.rename(
columns={
"timestamp":"date"
}
),
on="date",
how="left"
)

# Save the final combined master DataFrame to a CSV file
combined.to_csv(
"wildchat_combined_master.csv",
index=False
)

# Print a confirmation message
print(
"Final combined saved."
)

print("\nShape of combined master DataFrame:", combined.shape)
print("\nFirst 5 rows of combined master DataFrame:")
display(combined.head())

Final combined saved.

Shape of combined master DataFrame: (49550, 18)

First 5 rows of combined master DataFrame:


,conversation_id,conv_turn_count,avg_prompt_len,avg_sentiment,country,language,model,hashed_ip,toxicity_score,toxic_flag,drop_off_flag,response_quality_score,prompt_category_x,prompt_category_y,date,conversation_count,avg_tokens,toxicity_rate
0,000328f971437915462253ef7efa3d99,2,75.500,-0.038600,Russia,English,gpt-3.5-turbo-0301,03d3a8bcdcabd4bb19718415be4434e9260b6f5eafdfcf...,0.003256,0,1,8.664813,Coding,Coding,2026-04-27,NaN,NaN,NaN
1,000389c3eb8f1a4d2c4aa7867d647393,2,114.000,0.449650,Lithuania,English,gpt-4-0314,22a4f96891b72dd26f13cfc8da44b5e72d23b4aba516ce...,0.000277,0,1,9.489035,Other,Other,2026-04-27,NaN,NaN,NaN
2,0005a00b764ee8bec8e568719c979fe7,20,112.600,-0.040315,Italy,Italian,gpt-3.5-turbo-0301,332cd1b47fb8d0913529ab820a5bc616690cc1c68275e7...,0.004007,0,0,9.453345,Other,Other,2026-04-27,NaN,NaN,NaN
3,00070b104d8fdfdb538683c30de86654,4,405.000,-0.499825,Germany,German,gpt-3.5-turbo-0301,ec0e475c818e34ecd3a61e69737f0e85df44b59ec7df53...,0.015944,0,0,11.964874,Other,Other,2026-04-27,NaN,NaN,NaN
4,000712d3175ab7f12b9eec45da59b6b2,8,142.625,0.130975,Russia,Russian,gpt-4-0314,674476fe84e87fa33f374d2db98f909fd4322fb77919ff...,0.001905,0,0,9.928696,Casual,Casual,2026-04-27,NaN,NaN,NaN
